<style>

.fa-rtl {
  direction: rtl;
  text-align: right;
  font-family: Tahoma, Arial, sans-serif;
  line-height: 1.9;
  font-size: 16px;
}

.fa-rtl h1 {
  font-size: 30px;
}

.fa-rtl h2 {
  font-size: 24px;
}

.fa-rtl h3 {
  font-size: 20px;
}

.fa-rtl code {
  direction: ltr;
  unicode-bidi: isolate;
  font-family: Consolas, monospace;
}

.ltr {
  direction: ltr;
  unicode-bidi: isolate;
  display: inline-block;
}

mjx-container {
  direction: ltr !important;
  unicode-bidi: isolate !important;
  font-size: 112% !important;
}

mjx-container[display="true"] {
  text-align: center !important;
  margin: 1em 0 !important;
}

</style>

<div class="fa-rtl" dir="rtl">

# Machine Learning with Python — Session 3

## Lasso, Ridge, VIF & Mutual Information

**مدت:** ۲ ساعت  

**مجموعه‌داده:** <span class="ltr">Scikit-learn Diabetes Dataset</span>

### مسیر جلسه

۱. **Lasso Regression**

۲. تنظیم دستی `alpha` در Lasso

۳. **Ridge Regression**

۴. تنظیم دستی `alpha` در Ridge

۵. **Multicollinearity**

۶. **VIF**

۷. **Mutual Information (MI)**

</div>


<div class="fa-rtl" dir="rtl">



### در پایان جلسه

دانشجو باید بتواند:

- فرق Lasso و Ridge را توضیح دهد.
- مفهوم `alpha` را بفهمد.
- `alpha` را با Validation Set به‌صورت دستی امتحان کند.
- مفهوم Multicollinearity و VIF را توضیح دهد.
- با Mutual Information، Featureها را Rank کند.

</div>



<div class="fa-rtl" dir="rtl">

# 1. Setup

برای ساده ماندن کدها فقط Libraryهایی را Import می‌کنیم که واقعاً نیاز داریم.

</div>


In [1]:
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import mutual_info_regression

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [2]:
data = load_diabetes(as_frame=True, scaled=False)

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()


X shape: (442, 10)
y shape: (442,)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0



<div class="fa-rtl" dir="rtl">

## Train / Validation / Test

چون می‌خواهیم `alpha` را انتخاب کنیم، فقط Train/Test کافی نیست.

- **Train:** مدل یاد می‌گیرد.
- **Validation:** مقدار `alpha` را انتخاب می‌کنیم.
- **Test:** فقط در پایان استفاده می‌شود.

</div>


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=0
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=0
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (265, 10)
Validation: (88, 10)
Test: (89, 10)



<div class="fa-rtl" dir="rtl">

## Standardization

Lasso و Ridge به Scale Featureها حساس هستند.

پس `StandardScaler` را فقط روی Train یاد می‌گیریم و همان Scale را روی Validation و Test اعمال می‌کنیم.

</div>


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)



<div class="fa-rtl" dir="rtl">

# 2. مرور خیلی کوتاه Linear Regression

در Multiple Linear Regression:

</div>

$$
\hat{y}
=
\beta_0+\beta_1x_1+\beta_2x_2+\cdots+\beta_px_p
$$

<div class="fa-rtl" dir="rtl">

مدل تلاش می‌کند Predictionها به مقدار واقعی نزدیک باشند.

خطای مربعی مدل را می‌توانیم با RSS نشان دهیم:

</div>

$$
\mathrm{RSS}
=
\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$


In [6]:
linear_model = LinearRegression()

linear_model.fit(X_train_scaled, y_train)

linear_pred = linear_model.predict(X_val_scaled)

linear_mae = mean_absolute_error(y_val, linear_pred)

print("Linear Regression MAE:", round(linear_mae, 2))


Linear Regression MAE: 48.2



<div class="fa-rtl" dir="rtl">

# 3. Lasso Regression — L1 Regularization

Lasso همان Linear Regression است، اما یک **Penalty** هم اضافه می‌کند.

هدف:

- خطای مدل کم باشد.
- Coefficientها بیش از حد بزرگ نشوند.

</div>

$$
\mathrm{Lasso\ Loss}
=
\mathrm{RSS}
+
\alpha \sum_{j=1}^{p} |\beta_j|
$$

<div class="fa-rtl" dir="rtl">

### اجزای فرمول

**خطای مدل:**

</div>

$$
\mathrm{RSS}
$$

<div class="fa-rtl" dir="rtl">

**L1 Penalty — مجموع قدر مطلق Coefficientها:**

</div>

$$
\sum_{j=1}^{p} |\beta_j|
$$

<div class="fa-rtl" dir="rtl">

### معنی `alpha`

- `alpha` کوچک → Penalty ضعیف‌تر
- `alpha` بزرگ → Penalty قوی‌تر
- در Lasso بعضی Coefficientها می‌توانند **دقیقاً صفر** شوند.

پس Lasso می‌تواند به‌صورت خودکار بعضی Featureها را از مدل کنار بگذارد.

</div>



<div class="fa-rtl" dir="rtl">

## یک Lasso ساده

فعلاً فقط یک مقدار برای `alpha` می‌گذاریم تا Syntax را ببینیم.

</div>


In [7]:
lasso_model = Lasso(alpha=1)

lasso_model.fit(X_train_scaled, y_train)

lasso_pred = lasso_model.predict(X_val_scaled)

lasso_mae = mean_absolute_error(y_val, lasso_pred)

print("Lasso MAE:", round(lasso_mae, 2))


Lasso MAE: 48.16


In [8]:
lasso_coef = pd.DataFrame()

lasso_coef["Feature"] = X.columns
lasso_coef["Coefficient"] = lasso_model.coef_

lasso_coef


,Feature,Coefficient
0,age,-1.990374
1,sex,-9.288559
2,bmi,26.780435
3,bp,15.145874
4,s1,-9.572692
5,s2,-0.000000
6,s3,-10.473304
7,s4,0.000000
8,s5,30.787433
9,s6,1.049685



<div class="fa-rtl" dir="rtl">

## تنظیم دستی `alpha` در Lasso

چند مقدار را امتحان می‌کنیم.

مدلی را ترجیح می‌دهیم که **Validation MAE کمتری** داشته باشد.

> Test Set را برای انتخاب `alpha` استفاده نمی‌کنیم.

</div>


In [9]:
alpha_values = [0.01, 0.1, 1, 10]

for alpha in alpha_values:
    model = Lasso(alpha=alpha)
    model.fit(X_train_scaled, y_train)

    pred = model.predict(X_val_scaled)
    mae = mean_absolute_error(y_val, pred)

    print("alpha =", alpha, "   MAE =", round(mae, 2))


alpha = 0.01    MAE = 48.2
alpha = 0.1    MAE = 48.23
alpha = 1    MAE = 48.16
alpha = 10    MAE = 48.81



<div class="fa-rtl" dir="rtl">

### Quiz

اگر `alpha` در Lasso خیلی بزرگ شود، چه اتفاقی محتمل‌تر است؟

A) Coefficientها بزرگ‌تر می‌شوند  
B) Coefficientهای بیشتری کوچک یا صفر می‌شوند  
C) Penalty حذف می‌شود  

<details>
<summary><b>پاسخ</b></summary>

**B**

</details>

</div>



<div class="fa-rtl" dir="rtl">

# 4. Ridge Regression — L2 Regularization

Ridge هم به Linear Regression یک Penalty اضافه می‌کند.

تفاوت اصلی این است که Ridge از **مربع Coefficientها** استفاده می‌کند.

</div>

$$
\mathrm{Ridge\ Loss}
=
\mathrm{RSS}
+
\alpha \sum_{j=1}^{p} \beta_j^2
$$

<div class="fa-rtl" dir="rtl">

**L2 Penalty:**

</div>

$$
\sum_{j=1}^{p} \beta_j^2
$$

<div class="fa-rtl" dir="rtl">

### تفاوت اصلی

- **Lasso:** ممکن است بعضی Coefficientها را دقیقاً صفر کند.
- **Ridge:** Coefficientها را کوچک می‌کند، ولی معمولاً صفر نمی‌کند.

</div>


In [10]:
ridge_model = Ridge(alpha=1)

ridge_model.fit(X_train_scaled, y_train)

ridge_pred = ridge_model.predict(X_val_scaled)

ridge_mae = mean_absolute_error(y_val, ridge_pred)

print("Ridge MAE:", round(ridge_mae, 2))


Ridge MAE: 48.22



<div class="fa-rtl" dir="rtl">

## تنظیم دستی `alpha` در Ridge

دقیقاً مثل Lasso چند مقدار را امتحان می‌کنیم.

</div>


In [12]:
alpha_values = [0.01, 0.1, 1, 10, 100]

for alpha in alpha_values:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)

    pred = model.predict(X_val_scaled)
    mae = mean_absolute_error(y_val, pred)

    print("alpha =", alpha, "   MAE =", round(mae, 2))


alpha = 0.01    MAE = 48.2
alpha = 0.1    MAE = 48.2
alpha = 1    MAE = 48.22
alpha = 10    MAE = 48.08
alpha = 100    MAE = 47.74



<div class="fa-rtl" dir="rtl">

## Lasso vs Ridge

| ویژگی | Lasso | Ridge |
|---|---|---|
| Penalty | L1 | L2 |
| Coefficientها را کوچک می‌کند؟ | بله | بله |
| ممکن است Coefficient را صفر کند؟ | بله | معمولاً خیر |
| برای Feature Selection مناسب است؟ | می‌تواند باشد | مستقیم نه |

### یک راه ساده برای یادآوری

**Lasso → بعضی Featureها ممکن است حذف شوند**

**Ridge → همه را نگه می‌دارد ولی Coefficientها را کنترل می‌کند**

</div>



<div class="fa-rtl" dir="rtl">

# 5. Multicollinearity

Multicollinearity یعنی بعضی Predictorها اطلاعات بسیار مشابهی داشته باشند.

مثلاً اگر دو Feature خیلی شبیه هم حرکت کنند، Linear Regression ممکن است برای تقسیم اثر بین آن‌ها دچار مشکل شود.

یک بررسی ساده:

</div>


In [13]:
correlation = X_train["s1"].corr(X_train["s2"])

print("Correlation between s1 and s2:", round(correlation, 2))


Correlation between s1 and s2: 0.9



<div class="fa-rtl" dir="rtl">

# 6. VIF — Variance Inflation Factor

VIF بررسی می‌کند که یک Feature چقدر توسط **Featureهای دیگر** قابل توضیح است.

فرمول:

</div>

$$
\mathrm{VIF}_j
=
\frac{1}{1-R_j^2}
$$

<div class="fa-rtl" dir="rtl">

در این فرمول، \(R_j^2\) مربوط به مدلی است که Feature شماره \(j\) را با بقیه Featureها پیش‌بینی می‌کند.

### تفسیر ساده

- VIF نزدیک 1 → مشکل خاصی دیده نمی‌شود.
- VIF بزرگ‌تر → Multicollinearity بیشتر.
- عددهای خیلی بزرگ نیاز به بررسی دارند.

> VIF به‌تنهایی دستور حذف Feature نیست؛ فقط یک Diagnostic است.

</div>



<div class="fa-rtl" dir="rtl">

## محاسبه VIF

کد را عمداً ساده و مرحله‌به‌مرحله می‌نویسیم.

</div>


In [14]:
# Add a constant column so the regression includes an intercept. (import statsmodels.api as sm)
X_vif = sm.add_constant(X_train)

vif_numbers = []

for i in range(1, X_vif.shape[1]):
    value = variance_inflation_factor(X_vif.values, i)
    vif_numbers.append(value)

vif_table = pd.DataFrame()

vif_table["Feature"] = X_train.columns
vif_table["VIF"] = vif_numbers

vif_table = vif_table.sort_values("VIF", ascending=False)

vif_table


,Feature,VIF
4,s1,61.157652
5,s2,41.606553
6,s3,16.110399
8,s5,10.503437
7,s4,9.356066
2,bmi,1.578722
3,bp,1.564879
9,s6,1.549446
0,age,1.253300
1,sex,1.212365


In [16]:
# Drop s1
X_train_new = X_train.drop(columns=["s1"])

# Add constant
X_vif_new = sm.add_constant(X_train_new)

# Calculate VIF
vif_numbers = []

for i in range(len(X_train_new.columns)):
    vif = variance_inflation_factor(X_vif_new.values, i + 1)
    vif_numbers.append(vif)

# Create VIF table
vif_table_new = pd.DataFrame()

vif_table_new["Feature"] = X_train_new.columns
vif_table_new["VIF"] = vif_numbers

vif_table_new = vif_table_new.sort_values("VIF", ascending=False)

vif_table_new

,Feature,VIF
6,s5,1.720866
3,bp,1.546916
2,bmi,1.538820
7,s6,1.536433
5,s3,1.437895
0,age,1.250494
4,s2,1.230114
1,sex,1.206336



<div class="fa-rtl" dir="rtl">

### مثال سریع VIF

اگر:

</div>

$$
R_j^2=0.90
$$

<div class="fa-rtl" dir="rtl">

آنگاه:

</div>

$$
\mathrm{VIF}_j
=
\frac{1}{1-0.90}
=
10
$$

<div class="fa-rtl" dir="rtl">

یعنی آن Feature تا حد زیادی با کمک Featureهای دیگر قابل توضیح است.

</div>



<div class="fa-rtl" dir="rtl">

# 7. Mutual Information — MI

تا اینجا:

- Lasso و Ridge مدل بودند.
- VIF رابطه‌ی بین Predictorها را بررسی می‌کرد.

حالا MI می‌پرسد:

> هر Feature چقدر درباره Target اطلاعات می‌دهد؟

برای این جلسه لازم نیست فرمول ریاضی کامل MI را حفظ کنیم.

### تفسیر ساده

- MI بزرگ‌تر → Feature اطلاعات بیشتری درباره Target دارد.
- MI نزدیک صفر → وابستگی کمتری دیده شده است.
- MI می‌تواند رابطه‌های غیرخطی را هم پیدا کند.
- MI علامت مثبت/منفی رابطه را نشان نمی‌دهد.

</div>


In [17]:
mi_values = mutual_info_regression(
    X_train,
    y_train,
    random_state=0
)

mi_table = pd.DataFrame()

mi_table["Feature"] = X_train.columns
mi_table["MI"] = mi_values

mi_table = mi_table.sort_values("MI", ascending=False)

mi_table


,Feature,MI
8,s5,0.215163
2,bmi,0.169050
3,bp,0.135956
7,s4,0.127037
9,s6,0.098227
6,s3,0.064458
0,age,0.061370
4,s1,0.051196
1,sex,0.013245
5,s2,0.003612


<div class="fa-rtl" dir="rtl">

## انتخاب چند Feature با استفاده از MI

برای مثال، فقط **۵ Feature با بالاترین MI** را انتخاب می‌کنیم.

این انتخاب فقط برای Demonstration است و به این معنی نیست که عدد **۵** همیشه بهترین تعداد Feature است.

</div>

In [18]:
top_features = mi_table.head(5)["Feature"].tolist()

print("Top 5 features:")
print(top_features)


Top 5 features:
['s5', 'bmi', 'bp', 's4', 's6']



<div class="fa-rtl" dir="rtl">

# 8. Final Test

تا اینجا Test Set را برای انتخاب `alpha` استفاده نکردیم.

حالا یک مدل نهایی را فقط **یک بار** روی Test بررسی می‌کنیم.

برای سادگی این Notebook، از یکی از مقدارهای مناسب Ridge که در Validation دیدیم استفاده می‌کنیم.

 می‌توانید مقدار `alpha` را بر اساس خروجی Validation تغییر دهید.

</div>


In [19]:
final_model = Ridge(alpha=1)

final_model.fit(X_train_scaled, y_train)

test_pred = final_model.predict(X_test_scaled)

test_mae = mean_absolute_error(y_test, test_pred)

print("Final Test MAE:", round(test_mae, 2))


Final Test MAE: 42.96



<div class="fa-rtl" dir="rtl">

# 9. جمع‌بندی

### Lasso

</div>

$$
\mathrm{RSS}
+
\alpha\sum_{j=1}^{p}|\beta_j|
$$

<div class="fa-rtl" dir="rtl">

- L1 Regularization
- می‌تواند Coefficient را صفر کند.

### Ridge

</div>

$$
\mathrm{RSS}
+
\alpha\sum_{j=1}^{p}\beta_j^2
$$

<div class="fa-rtl" dir="rtl">

- L2 Regularization
- Coefficientها را Shrink می‌کند.
- معمولاً آن‌ها را صفر نمی‌کند.

### VIF

</div>

$$
\mathrm{VIF}_j
=
\frac{1}{1-R_j^2}
$$

<div class="fa-rtl" dir="rtl">

- Diagnostic برای Multicollinearity

### MI

- ارتباط Feature با Target را Rank می‌کند.
- فقط محدود به رابطه Linear نیست.

</div>



<div class="fa-rtl" dir="rtl">

# Exit Quiz

1. تفاوت اصلی Lasso و Ridge چیست؟
2. `alpha` بزرگ‌تر چه اثری روی Regularization دارد؟
3. چرا Featureها را قبل از Lasso/Ridge Standardize کردیم؟
4. چرا `alpha` را با Test Set انتخاب نمی‌کنیم؟
5. VIF چه چیزی را بررسی می‌کند؟
6. آیا VIF از Target استفاده می‌کند؟
7. MI چه چیزی را بررسی می‌کند؟
8. آیا MI علامت مثبت یا منفی رابطه را نشان می‌دهد؟

<details>
<summary><b>پاسخ‌ها</b></summary>

1. Lasso از L1 و Ridge از L2 استفاده می‌کند؛ Lasso می‌تواند بعضی Coefficientها را صفر کند.
2. Penalty قوی‌تر می‌شود.
3. تا Scale Featureها باعث رفتار ناعادلانه Penalty نشود.
4. چون Test باید برای ارزیابی نهایی دست‌نخورده بماند.
5. میزان Multicollinearity بین Predictorها.
6. خیر.
7. میزان اطلاعات/وابستگی Feature نسبت به Target.
8. خیر.

</details>

</div>


<div class="fa-rtl" dir="rtl">

### تمرین Lasso

چند مقدار مختلف برای `alpha` در مدل Lasso امتحان کنید.

برای هر مقدار:

1. مدل را روی Training Data آموزش دهید.
2. MAE را روی Validation Data محاسبه کنید.
3. بررسی کنید با تغییر `alpha`، عملکرد مدل چه تغییری می‌کند.
4. بهترین مقدار `alpha` را بر اساس کمترین Validation MAE انتخاب کنید.

در پایان گزارش کنید:

- چه مقدارهایی از `alpha` را امتحان کردید؟
- بهترین `alpha` چه بود؟
- Validation MAE برای بهترین مدل چقدر شد؟

</div>